# F1 Telemetry Analysis

Compare two drivers' laps from any Formula 1 session using [FastF1](https://docs.fastf1.dev/): speed, throttle, brake, gear, RPM, a steering estimate, delta time, and corner-by-corner performance -- for **any circuit on the calendar**, not just a hard-coded one.

Set the session and drivers in the config cell below, then **Run All**.

In [ ]:
from f1_telemetry import AnalysisConfig
from f1_telemetry.data_loader import load_session, select_lap
from f1_telemetry.telemetry import build_synced_telemetry
from f1_telemetry.corners import get_corner_locations, build_corner_analysis, find_biggest_advantage
from f1_telemetry.visualization import plot_telemetry_dashboard, plot_track_map
from f1_telemetry.report import build_sector_table, build_summary_stats, generate_race_report


## 1. Configuration

Edit these values for any session and any two drivers.

In [ ]:
config = AnalysisConfig(
    year=2025,
    event="Monaco Grand Prix",
    session="Q",           # "FP1", "FP2", "FP3", "Q", "SQ", or "R"
    driver1="VER",
    driver2="LEC",
    driver1_lap="Fastest", # or an explicit lap number, e.g. 18
    driver2_lap="Fastest",
)
config


## 2. Load session data

Downloads (and locally caches) lap and telemetry data for the session.

In [ ]:
session = load_session(config)
print(session)
print(f"Total laps recorded: {len(session.laps)}")


In [ ]:
lap1 = select_lap(session, config.driver1, config.driver1_lap)
lap2 = select_lap(session, config.driver2, config.driver2_lap)

print(f"{config.driver1}: Lap {lap1['LapNumber']:.0f}  Time {lap1['LapTime']}")
print(f"{config.driver2}: Lap {lap2['LapNumber']:.0f}  Time {lap2['LapTime']}")


## 3. Build synchronised telemetry

Resamples both laps onto a shared distance axis so they can be compared point-for-point, and derives delta time, a steering estimate, and a wheel-speed estimate.

In [ ]:
telemetry = build_synced_telemetry(lap1, lap2, config.driver1, config.driver2)
print("Telemetry synchronised:", len(telemetry.distance), "points")


## 4. Corner locations

Pulled from FastF1's circuit info -- this works for any track on the calendar, no manual corner list required.

In [ ]:
corner_locations = get_corner_locations(session)
corner_locations


## 5. Telemetry dashboard

An interactive, linked multi-channel view: speed, RPM, gear, throttle, brake, steering estimate, wheel speed estimate, and delta time, with corner markers overlaid.

In [ ]:
dashboard = plot_telemetry_dashboard(
    telemetry,
    config.driver1,
    config.driver2,
    corner_locations=corner_locations,
    title=f"{config.event} {config.session} - {config.driver1} vs {config.driver2}",
)
dashboard.show()


## 6. Track map

In [ ]:
pos1 = lap1.get_pos_data().add_distance()
pos2 = lap2.get_pos_data().add_distance()

fig = plot_track_map(
    pos1, config.driver1, config.event,
    pos2=pos2, driver2=config.driver2,
    corner_locations=corner_locations,
)
fig.show()


## 7. Corner-by-corner analysis

Entry / apex / exit speed, brake point, and throttle application at every corner, for both drivers.

In [ ]:
corner_df = build_corner_analysis(corner_locations, telemetry, config.driver1, config.driver2)
corner_df


## 8. Sector times and summary stats

In [ ]:
sector_table = build_sector_table(lap1, lap2, config.driver1, config.driver2)
sector_table


In [ ]:
summary_stats = build_summary_stats(telemetry, config.driver1, config.driver2)
summary_stats


## 9. Race engineer report

In [ ]:
biggest_advantage = find_biggest_advantage(corner_df, config.driver1, config.driver2, metric="EntrySpeed")

report_text = generate_race_report(
    config, lap1, lap2, telemetry, sector_table, biggest_advantage
)
print(report_text)


## 10. Export outputs

In [ ]:
dashboard.write_html("TelemetryDashboard.html")

with open("RaceEngineerReport.txt", "w") as f:
    f.write(report_text)

print("Saved TelemetryDashboard.html and RaceEngineerReport.txt")
